# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [1]:
#!pip install -qU ragas==0.2.10

In [2]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [3]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/anantabastola/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/anantabastola/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [5]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [6]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [7]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [8]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [9]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/Users/anantabastola/learn/AI_Makerspace/first_week/AIE7/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/Users/anantabastola/learn/AI_Makerspace/first_week/AIE7/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/Users/anantabastola/learn/AI_Makerspace/first_week/AIE7/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [10]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [12]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 40, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [13]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/34 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/40 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/62 [00:00<?, ?it/s]

Property 'summary' already exists in node 'd0087a'. Skipping!
Property 'summary' already exists in node '8971d2'. Skipping!
Property 'summary' already exists in node '971dae'. Skipping!
Property 'summary' already exists in node 'd08538'. Skipping!
Property 'summary' already exists in node 'da080c'. Skipping!
Property 'summary' already exists in node 'c97911'. Skipping!
Property 'summary' already exists in node '86356d'. Skipping!
Property 'summary' already exists in node '624a4b'. Skipping!
Property 'summary' already exists in node '7b441c'. Skipping!
Property 'summary' already exists in node 'd49ac9'. Skipping!
Property 'summary' already exists in node 'f063f4'. Skipping!
Property 'summary' already exists in node '44ed2d'. Skipping!
Property 'summary' already exists in node '56711a'. Skipping!
Property 'summary' already exists in node '515317'. Skipping!
Property 'summary' already exists in node '64332c'. Skipping!
Property 'summary' already exists in node 'cc9980'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/12 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/86 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'd0087a'. Skipping!
Property 'summary_embedding' already exists in node '971dae'. Skipping!
Property 'summary_embedding' already exists in node '8971d2'. Skipping!
Property 'summary_embedding' already exists in node 'd08538'. Skipping!
Property 'summary_embedding' already exists in node 'c97911'. Skipping!
Property 'summary_embedding' already exists in node 'da080c'. Skipping!
Property 'summary_embedding' already exists in node '86356d'. Skipping!
Property 'summary_embedding' already exists in node '624a4b'. Skipping!
Property 'summary_embedding' already exists in node 'f063f4'. Skipping!
Property 'summary_embedding' already exists in node '7b441c'. Skipping!
Property 'summary_embedding' already exists in node 'd49ac9'. Skipping!
Property 'summary_embedding' already exists in node '44ed2d'. Skipping!
Property 'summary_embedding' already exists in node '56711a'. Skipping!
Property 'summary_embedding' already exists in node '515317'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 80, relationships: 1939)

We can save and load our knowledge graphs as follows.

In [14]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 80, relationships: 1939)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [15]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [16]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

💡 SingleHopSpecificQuerySynthesizer: Generates simple, fact-based questions answered with on sentence. Example: “Where is the Eiffel Tower?”

💡 MultiHopAbstractQuerySynthesizer: Creates complex, conceptual questions needing multiple ideas. Example: “How did 20th-century wars affect diplomacy?”

💡 MultiHopSpecificQuerySynthesizer: Makes multi-step factual questions combining several facts. Example: “Who discovered radium and won a Nobel Prize?”



Finally, we can use our `TestSetGenerator` to generate our testset!

In [17]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is department,"[Chapter 1 Academic Years, Academic Calendars,...",The context does not specify a particular defi...,single_hop_specifc_query_synthesizer
1,What does 34 CFR 668.3(b) mean in terms of wee...,[Regulatory Citations Academic year minimums: ...,Regulatory Citations Academic year minimums ar...,single_hop_specifc_query_synthesizer
2,What is the significance of Chapter 3 in the c...,[Inclusion of Clinical Work in a Standard Term...,Inclusion of clinical work in a standard term ...,single_hop_specifc_query_synthesizer
3,What is Title IV and how does it relate to pay...,[Non-Term Characteristics A program that measu...,Title IV programs are subject to payment perio...,single_hop_specifc_query_synthesizer
4,Could you explain how the FSEOG program determ...,[both the credit or clock hours and the weeks ...,"The disbursement of FSEOG funds, like Pell Gra...",single_hop_specifc_query_synthesizer
5,Considering the definitions and characteristic...,[<1-hop>\n\nInclusion of Clinical Work in a St...,A program that measures student progress in cl...,multi_hop_abstract_query_synthesizer
6,"How do program requirements, such as minimum w...","[<1-hop>\n\nChapter 1 Academic Years, Academic...",Program requirements specify that for both und...,multi_hop_abstract_query_synthesizer
7,How do the criteria for successful program com...,[<1-hop>\n\nboth the credit or clock hours and...,The criteria for successful program completion...,multi_hop_abstract_query_synthesizer
8,school have different academic years for progr...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",A school may have different academic years for...,multi_hop_specific_query_synthesizer
9,Volume 8 and Volume 7 are related how they aff...,[<1-hop>\n\nboth the credit or clock hours and...,Volume 8 explains how the timing of disburseme...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [18]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '734510'. Skipping!
Property 'summary' already exists in node '82e000'. Skipping!
Property 'summary' already exists in node '0b338e'. Skipping!
Property 'summary' already exists in node '3376c0'. Skipping!
Property 'summary' already exists in node '0c7f70'. Skipping!
Property 'summary' already exists in node '0049df'. Skipping!
Property 'summary' already exists in node '584475'. Skipping!
Property 'summary' already exists in node '933e3a'. Skipping!
Property 'summary' already exists in node '89b4e3'. Skipping!
Property 'summary' already exists in node 'ae1663'. Skipping!
Property 'summary' already exists in node '2c114e'. Skipping!
Property 'summary' already exists in node '225395'. Skipping!
Property 'summary' already exists in node '6c50bd'. Skipping!
Property 'summary' already exists in node '79664f'. Skipping!


...

Property 'summary_embedding' already exists in node '734510'. Skipping!
Property 'summary_embedding' already exists in node '82e000'. Skipping!
Property 'summary_embedding' already exists in node '584475'. Skipping!
Property 'summary_embedding' already exists in node '79664f'. Skipping!
Property 'summary_embedding' already exists in node '933e3a'. Skipping!
Property 'summary_embedding' already exists in node '2c114e'. Skipping!
Property 'summary_embedding' already exists in node '0049df'. Skipping!
Property 'summary_embedding' already exists in node '0c7f70'. Skipping!
Property 'summary_embedding' already exists in node '0b338e'. Skipping!
Property 'summary_embedding' already exists in node '3376c0'. Skipping!
Property 'summary_embedding' already exists in node '6c50bd'. Skipping!
Property 'summary_embedding' already exists in node 'ae1663'. Skipping!
Property 'summary_embedding' already exists in node '225395'. Skipping!
Property 'summary_embedding' already exists in node '89b4e3'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [19]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How do disbursements work in the context of fe...,"[Chapter 1 Academic Years, Academic Calendars,...",Disbursements refer to the payments made to st...,single_hop_specifc_query_synthesizer
1,What does 34 CFR 668.3(a) specify regarding th...,[Regulatory Citations Academic year minimums: ...,Regulatory citations indicate that 34 CFR 668....,single_hop_specifc_query_synthesizer
2,What is the significance of Chapter 3 in relat...,[Inclusion of Clinical Work in a Standard Term...,Inclusion of clinical work in a standard term ...,single_hop_specifc_query_synthesizer
3,Can you explain how the FWS program differs fr...,[Non-Term Characteristics A program that measu...,The payment period is applicable to all Title ...,single_hop_specifc_query_synthesizer
4,If a student is in a clock-hour or non-term cr...,[<1-hop>\n\nboth the credit or clock hours and...,In clock-hour or non-term credit-hour programs...,multi_hop_abstract_query_synthesizer
5,How do overlapping courses across nonstandard ...,[<1-hop>\n\nInclusion of Clinical Work in a St...,"Overlapping courses across nonstandard terms, ...",multi_hop_abstract_query_synthesizer
6,How does the credit hour allocation for clinic...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The inclusion of clinical work in a standard t...,multi_hop_abstract_query_synthesizer
7,How does the control over clinical work schedu...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The inclusion of clinical work outside standar...,multi_hop_abstract_query_synthesizer
8,Volume 8 include clinical work in standard ter...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Volume 8 explains that clinical work can be in...,multi_hop_specific_query_synthesizer
9,How do Appendix A and Appendix B relate to the...,[<1-hop>\n\nDisbursement Timing in Subscriptio...,Appendix B provides detailed guidance on the d...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [20]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [21]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [22]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [23]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [24]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [25]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [26]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [27]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [28]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [29]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [30]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'The kinds of loans available are:\n\n- Direct PLUS Loan or student Federal PLUS Loan  \n- Direct Subsidized Loan  \n- Direct Unsubsidized Loan  \n- Subsidized and Unsubsidized Federal Stafford Loans (made under the FFEL Program before July 1, 2010)  \n- Federal SLS Loans  \n- Federal PLUS Loans (made under the FFEL Program before July 1, 2010)'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [31]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [32]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`: Evaluates answer correctness — checks if the model's response matches the correct answer (basic Q&A accuracy).
- `labeled_helpfulness_evaluator`: Evaluates helpfulness — judges how helpful the response is compared to the reference answer.
- `empathy_evaluator`:Evaluates empathy — checks if the response feels emotionally supportive and understanding to the user.

## LangSmith Evaluation

In [33]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'brief-stove-44' at:
https://smith.langchain.com/o/d271d21b-9aba-41e9-ab3b-0c1eb6900d84/datasets/44e2c193-c4c7-4f13-a04a-cdbc2829c4cd/compare?selectedSessions=454d0e55-d1b0-48fd-9d53-a429f41409a5




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do Chapters 2 and 3 relate to the inclusio...,I don't know.,None,Chapter 2 discusses the criteria for including...,0,0,0,0.899871,55c2d07e-1d32-4e79-aead-c1a812e09c06,a0d3c3ef-37e4-46b1-9d98-0237542c0223
1,"In volume 2 and volume 7, how does the academi...",Based on the provided context from the documen...,None,"In volume 2, it explains that academic years m...",1,1,0,8.959236,0e3603f5-8541-4663-a62d-5741722377c7,e2811a7a-35a4-4da4-a0b8-d8a2c893817c
2,How do Appendix A and Appendix B relate to the...,I don't know.,None,Appendix B provides detailed guidance on the d...,0,0,0,0.870735,8739d3f2-7655-48eb-b8c8-93c96c066317,49352807-e9ef-4b24-9538-1daa02f7b2a1
3,Volume 8 include clinical work in standard ter...,I don't know.,None,Volume 8 explains that clinical work can be in...,0,0,0,1.030843,879265ad-7ab7-4ae8-bb11-8d2fdf06c9f2,b856ddcc-ba89-4e00-87ef-aac9d79b8b66
4,How does the control over clinical work schedu...,"Based on the provided context, terms that incl...",None,The inclusion of clinical work outside standar...,1,0,0,3.993891,aa6d5d22-fc85-42f2-8c16-fb0b9af99d8f,6fcd545e-45c4-4dc4-9bdf-9941d8614649
5,How does the credit hour allocation for clinic...,"Based on the provided context, clinical work t...",None,The inclusion of clinical work in a standard t...,1,1,0,5.746303,2c83001c-ed27-47c7-8a2c-a06b6e3d5db1,42ed29db-a8a7-4c89-8313-cbea7e983270
6,How do overlapping courses across nonstandard ...,I don't know.,None,"Overlapping courses across nonstandard terms, ...",0,0,0,0.846637,6c230779-c21c-4638-af70-4f51c3e203da,2c294809-b616-4093-bcc3-9f7b3cf4ee14
7,If a student is in a clock-hour or non-term cr...,Based on the provided context:\n\n- Completion...,None,In clock-hour or non-term credit-hour programs...,1,1,0,4.299871,bdceeb63-e2bc-474e-98fd-40ebb66dc5e9,718f2357-9f7a-49ab-bca4-7ab51e542db3
8,Can you explain how the FWS program differs fr...,"Based on the provided context, the Federal Wor...",None,The payment period is applicable to all Title ...,1,1,0,1.921113,8e1d8937-5557-4cc7-9b6d-36b467488774,9934ebb3-dc10-463d-af22-b66decfad222
9,What is the significance of Chapter 3 in relat...,Chapter 3 provides information on BBAY (Borrow...,None,Inclusion of clinical work in a standard term ...,0,0,0,2.110259,18869274-5774-4f2b-89b0-48e567f701a6,e7245a30-9d28-4696-abfc-087caca778d4


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [34]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [35]:
rag_documents = docs

In [36]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

Larger chunks will result in better context, fewer retrieval calls and they are better for multi-hop questions.
Smaller shunks will give us more precise retreival, better for simple factual questions and reduce noise.

In [37]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

💡 Modifying the embedding model affects performance because better embeddings improve how well the system understands and matches meaning—leading to more accurate retrieval, fewer irrelevant results, and better answers from the LLM.

In [38]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [39]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [40]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [41]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for your question—it's really important to understand the options available when it comes to student loans, and I'm here to help clarify this for you.\n\nBased on the information provided in the context, there are several types of federal student loans available:\n\n1. **Direct Subsidized Loans**: These loans are based on the student's financial need. The federal government pays the interest on these loans while the student is in school at least half-time, during the grace period, and during deferment periods.\n\n2. **Direct Unsubsidized Loans**: These loans are not based on financial need, and interest accrues during all periods. Both dependent and independent students may be eligible for these loans.\n\n3. **Direct PLUS Loans**: These loans are available to the parents of dependent students (Parent PLUS Loans) or to graduate/professional students (Grad PLUS Loans). They can cover up to the student's cost of attendance minus other financial aid received. There is no fixed l

Finally, we can evaluate the new chain on the same test set!

In [42]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'slight-field-84' at:
https://smith.langchain.com/o/d271d21b-9aba-41e9-ab3b-0c1eb6900d84/datasets/44e2c193-c4c7-4f13-a04a-cdbc2829c4cd/compare?selectedSessions=cc978f23-5413-45f6-b5a0-2febe8705546




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do Chapters 2 and 3 relate to the inclusio...,Thank you for your thoughtful question. Based ...,None,Chapter 2 discusses the criteria for including...,1,1,1,3.847833,55c2d07e-1d32-4e79-aead-c1a812e09c06,211f9330-a83a-4dfc-8c68-e148246f39c7
1,"In volume 2 and volume 7, how does the academi...",Thank you for your thoughtful question. I unde...,None,"In volume 2, it explains that academic years m...",1,0,1,11.980084,0e3603f5-8541-4663-a62d-5741722377c7,c630a354-1edf-4db7-b493-0c34f28ff24c
2,How do Appendix A and Appendix B relate to the...,Thank you for your thoughtful question. From t...,None,Appendix B provides detailed guidance on the d...,1,0,1,3.487849,8739d3f2-7655-48eb-b8c8-93c96c066317,7303664f-1de1-4d40-8c96-152d2b360c3b
3,Volume 8 include clinical work in standard ter...,Thank you for your thoughtful question. Based ...,None,Volume 8 explains that clinical work can be in...,1,1,1,2.753755,879265ad-7ab7-4ae8-bb11-8d2fdf06c9f2,b2b2f196-f5e4-4db7-9277-aa68405360a8
4,How does the control over clinical work schedu...,Thank you for your thoughtful question. Based ...,None,The inclusion of clinical work outside standar...,1,1,1,4.334241,aa6d5d22-fc85-42f2-8c16-fb0b9af99d8f,d3439cad-43d0-47f1-bd25-8f9a1997a34a
5,How does the credit hour allocation for clinic...,Thank you for your thoughtful question. It sou...,None,The inclusion of clinical work in a standard t...,1,1,1,7.365690,2c83001c-ed27-47c7-8a2c-a06b6e3d5db1,8b9e1709-eb42-49d5-97eb-064fa568b29a
6,How do overlapping courses across nonstandard ...,Thank you for your thoughtful question. Based ...,None,"Overlapping courses across nonstandard terms, ...",1,1,1,10.208864,6c230779-c21c-4638-af70-4f51c3e203da,838eb507-f28c-4234-a97c-bbe12321765f
7,If a student is in a clock-hour or non-term cr...,Thank you for your thoughtful question. It sou...,None,In clock-hour or non-term credit-hour programs...,1,1,1,7.394863,bdceeb63-e2bc-474e-98fd-40ebb66dc5e9,c404899a-c731-41d9-b81c-b4bd0b79e130
8,Can you explain how the FWS program differs fr...,Thank you for your question—it's completely un...,None,The payment period is applicable to all Title ...,1,1,1,3.522217,8e1d8937-5557-4cc7-9b6d-36b467488774,4a3a63b3-72cf-4fa0-be73-00ff979db22c
9,What is the significance of Chapter 3 in relat...,Thank you for your thoughtful question. Based ...,None,Inclusion of clinical work in a standard term ...,0,0,1,2.613053,18869274-5774-4f2b-89b0-48e567f701a6,334438de-ee59-416b-a99d-36bc9fb290b5


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.


Answer:

- brief-stove-44
1. chunk-size: 500
2. embedding mode: `text-embedding-3-small`

- slight-field-84,  chunk-size: 1000
1. chunk-size: 1000
2. embedding mode: `text-embedding-3-large`


![compare](./compare.png)
![compare-2](./compare-2.png)


1. Correctness Metric:
 - **brief-stove-44**: 0.58 🟥
 - **slight-field-84**: 0.91 🟩 
 Using larger chunks (1000 vs 500) reduced info fragmentation and gave more complete context. Upgrading to the larger embedding model (3-large) improved semantic understanding. 


2. Empathy Metric:
 - **brief-stove-44**: 0.0 🟥
 - **slight-field-84**: 1.0 🟩
 Bigger chunks captured full emotional context. Better embeddings picked up on subtle emotional cues.


3. Helpfulness Metric:
 - **brief-stove-44**: 0.5 🟥
 - **slight-field-84**: 0.75 🟩
 Better context led to more helpful suggestions






